In [59]:
# Importy i ustawienie katalogu głównego projektu
import sys, os, re
import pandas as pd
import numpy as np
from pathlib import Path

# Przejdź do katalogu głównego projektu tylko raz — bezpieczne przy wielokrotnym uruchomieniu
if os.path.basename(os.getcwd()) != "analiza_sprzedazy_w_sklepie":
    os.chdir("../..")

sys.path.insert(0, os.getcwd())
import utils.tools as tools


In [60]:
# Suma kontrolna wejścia — ten sam hash u obu osób = identyczne dane
nazwa_pliku = "fact_inka_calendar.parquet"
moj_hash = tools.hash_danych_bezpieczny(f"dane/interim/{nazwa_pliku}")
print(f"Mój hash (posortowane):   {nazwa_pliku}   {moj_hash}")

Mój hash (posortowane):   fact_inka_calendar.parquet   dd831a813d662099be81de7050891487f7e95808cf0c2aad19404e7a4484ef6e


# Flagi towarowe
**CzyWazone — towar sprzedawany na wagę**

In [61]:
# Wczytanie kalendarza (TowId × Data + ruchy magazynowe) do df_flagi
df_flagi = pd.read_parquet(f"dane/interim/{nazwa_pliku}")

In [62]:
# Zamiana typów kolumn 
df_flagi['Producent'] = df_flagi['Producent'].fillna(-1).astype('int16')  
df_flagi = df_flagi.astype({
    'TypDok': 'int32',
    'TypPoz': 'int32',
    'DokId': 'int32',
    'AsId': 'int32',
    'Kolejnosc': 'int32',
    'NrPozycji': 'int32',
    'KolejnyWDniu': 'int16',
    'AktywnyDok': 'int8',
    'AktywnyTow': 'int8',
    'Mnoznik': 'int8',
    'WplywNaStan': 'int8',
    'CzyNiechciane': 'int8',    
    'Mnoznik': 'int8',

    
})
df_flagi['EAN'] = df_flagi['EAN'].astype('int64').astype(str).astype('category')


### Towary wazone 

In [63]:
# ============================================================
# SOFT FILTER — Kryterium 2: Towary ważone (per TowId)
# Krok 1: % transakcji z ułamkową ilością per TowId
# ============================================================

# LiczbaTransakcji = wszystkie wiersze towaru (też puste dni i ruchy), LiczbaUlamkowych = IloscPlus z ułamkiem
wazone_check = df_flagi.groupby('TowId').agg(
    LiczbaTransakcji=('IloscPlus', 'count'),
    LiczbaUlamkowych=('IloscPlus', lambda x: (x % 1 != 0).sum()),
).reset_index()

wazone_check['PctUlamkowych'] = (
    wazone_check['LiczbaUlamkowych'] / wazone_check['LiczbaTransakcji'] * 100
)

print(wazone_check['PctUlamkowych'].describe())

count    12481.000000
mean         0.675678
std          6.449838
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max         97.207528
Name: PctUlamkowych, dtype: float64


In [64]:
# Wykrywanie wagi po nazwie: samodzielne 'KG' (bez liczby przed nim) = towar na wagę
def czy_nazwa_sugeruje_wage(nazwa):
    nazwa = str(nazwa).upper().strip()
    
    # Wyklucz przypadki "liczba+KG" (stała gramatura opakowania, np. "0,5KG", "1,65KG")
    bez_gramatury = re.sub(r'\d+[\.,]?\d*\s*KG\b', '', nazwa)
    
    # Szukamy samodzielnego "KG" jako jednostki sprzedaży (bez liczby bezpośrednio przed nim)
    return bool(re.search(r'\bKG\b', bez_gramatury))

# Szybki test kontrolny
testy = [
    "Chleb mieszany 0.6 kg PRECELEK",         # False - gramatura
    "PROSZEK PERSIL UNIVERSAL 1,65KG HENKEL", # False - gramatura
    "MĄKA ZIEMNIACZANA 0,5KG",                # False - gramatura
    "SURÓWKA KG",                              # True - waga
    "BAKŁAŻAN KG POLSKA",                      # True - waga
    "SAŁATKA WIELKANOCNA KG GRZEŚKOWIAK",      # True - waga
]
for t in testy:
    print(f"{czy_nazwa_sugeruje_wage(t)}: {t}")

False: Chleb mieszany 0.6 kg PRECELEK
False: PROSZEK PERSIL UNIVERSAL 1,65KG HENKEL
False: MĄKA ZIEMNIACZANA 0,5KG
True: SURÓWKA KG
True: BAKŁAŻAN KG POLSKA
True: SAŁATKA WIELKANOCNA KG GRZEŚKOWIAK


In [65]:
# Finalna flaga JestWazony per TowId: >= 50% ułamkowych ilości LUB nazwa z samodzielnym 'KG'
nazwy_do_sprawdzenia = df_flagi[['TowId', 'NazwaTow']].drop_duplicates(subset='TowId')
nazwy_do_sprawdzenia['NazwaSugerujeWage'] = nazwy_do_sprawdzenia['NazwaTow'].apply(czy_nazwa_sugeruje_wage)

wazone_check_pelne = wazone_check.merge(nazwy_do_sprawdzenia, on='TowId', how='left')

wazone_check_pelne['JestWazony'] = (
    (wazone_check_pelne['PctUlamkowych'] >= 50) |
    (wazone_check_pelne['NazwaSugerujeWage'])
)

lista_wazonych_towid = set(wazone_check_pelne[wazone_check_pelne['JestWazony']]['TowId'])
print(f"Ważonych TowId (finalne kryterium): {len(lista_wazonych_towid)}")

dodane_przez_nazwe = wazone_check_pelne[
    (wazone_check_pelne['NazwaSugerujeWage']) & 
    (wazone_check_pelne['PctUlamkowych'] < 50)
]
print(f"\nDodatkowo złapane przez samodzielne 'KG' (bez liczby): {len(dodane_przez_nazwe)}")
print(dodane_przez_nazwe[['TowId', 'NazwaTow', 'PctUlamkowych']].sort_values('NazwaTow'))

Ważonych TowId (finalne kryterium): 193

Dodatkowo złapane przez samodzielne 'KG' (bez liczby): 118
       TowId                           NazwaTow  PctUlamkowych
4988   51262                          ANANAS KG      11.165644
5390   52141                 BAKŁAŻAN KG POLSKA       9.375000
3564   25652      BRZUSZKI Z ŁOSOSIA WĘDZONE KG       8.280255
4382   49625                          BURAKI KG      11.061552
9166   77611      CIASTKA MARKIZY KG  DR GERARD      10.152284
...      ...                                ...            ...
2735   16441              SER SALAMI MLEKPOL KG      46.389646
2748   16479   SER Z ORZECHEM WŁOSKIM KG ŁOWICZ      47.916667
11332  80088                         SURÓWKA KG      11.764706
5853   53068  USZKA Z GRZYBAMI KG KUCHNIA POLKI       2.726146
8870   67234               ZIEMNIAKI SŁODKIE KG      34.860558

[118 rows x 3 columns]


In [66]:
# Przypisanie JestWazony do wierszy + kontrola, z jakich asortymentów pochodzą ważone towary
df_flagi['JestWazony'] = df_flagi['TowId'].isin(lista_wazonych_towid).astype('int8')

print(df_flagi['JestWazony'].value_counts())

nazwy_wazonych = (
    df_flagi[df_flagi['TowId'].isin(lista_wazonych_towid)]
    [['TowId', 'NazwaTow', 'NazwaAsort']]
    .drop_duplicates(subset='TowId')
)
print(nazwy_wazonych['NazwaAsort'].value_counts())

JestWazony
0    9118366
1     367506
Name: count, dtype: int64
NazwaAsort
CUKIERKI WAGA                    42
SERY WAGA /nabiał                33
WARZYWA                          31
OWOCE                            19
CUKIERKI                         16
WĘDLINY WAGA                     10
CIASTKA WAGA                      8
PIEROGI KOPYTKA KROKIETY INNE     6
WĘDLINY PACZKOWANE                4
RYBY WĘDZONE WAGA /ryby           3
WARZYWA KISZONE                   3
PIECZYWO                          2
RYBY MROŻONE WAGA                 2
PRZETWORY RYBNE /ryby             2
MARKA WŁASNA SPAR                 2
KAWY                              2
SURÓWKI I SAŁATKI                 2
MIĘSO DROBIOWE WAGA               1
MIĘSO, WĘDLINY I GARMAŻERKA       1
WIELKANOCNE                       1
***PRZECENY                       1
ŚWIĄTECZNE                        1
CIASTKA                           1
Name: count, dtype: int64


## CzyMartwy — brak sprzedaży w ostatnich N dniach 

In [67]:
# ============================================================
# CzyMartwy — z uwzględnieniem sezonowości (próg per-TowId)
# Liczone wyłącznie na SPRZEDAŻY (TypDok == 21, DokId != -1).
# Ruchy PZ / BO / REM itd. nie "ożywiają" towaru.
# ============================================================

df_flagi = df_flagi.sort_values(['TowId', 'Data']).reset_index(drop=True)

# Maska: realna sprzedaż (puste dni mają DokId = -1, inne ruchy mają TypDok != 21)
jest_sprzedaz = (df_flagi['DokId'] != -1) & (df_flagi['TypDok'] == 21)

# 1) Data ostatniej sprzedaży (forward-fill w obrębie TowId)
df_flagi['DataOstatniejSprzedazy'] = df_flagi['Data'].where(jest_sprzedaz)
df_flagi['DataOstatniejSprzedazy'] = (
    df_flagi.groupby('TowId')['DataOstatniejSprzedazy'].ffill()
)

df_flagi['DniOdSprzedazy'] = (
    df_flagi['Data'] - df_flagi['DataOstatniejSprzedazy']
).dt.days.astype('Int16')  

# 2) Historyczne przerwy między KOLEJNYMI dniami sprzedaży, per TowId
dni_sprzedazy = (
    df_flagi.loc[jest_sprzedaz, ['TowId', 'Data']]
    .drop_duplicates()
    .sort_values(['TowId', 'Data'])
)
dni_sprzedazy['PoprzedniaData'] = dni_sprzedazy.groupby('TowId')['Data'].shift(1)
dni_sprzedazy['Przerwa'] = (dni_sprzedazy['Data'] - dni_sprzedazy['PoprzedniaData']).dt.days

# 3) Własny "rekord" normalnej przerwy każdego TowId (95 percentyl — odporny na pojedynczy wyjątek)
prog_indywidualny = (
    dni_sprzedazy.groupby('TowId')['Przerwa']
    .quantile(0.95)
    .rename('PrzerwaHistoryczna95pct')
)

# Data ostatniej sprzedaży per TowId (do flagi "na koniec danych")
ostatnia_sprzedaz = dni_sprzedazy.groupby('TowId')['Data'].max()

# 4) Próg per TowId = margines nad własną historią, z podłogą dla towarów z krótką historią.
#    Towary z jednym dniem sprzedaży nie mają przerwy (NaN) -> dostają samą podłogę.
N_MINIMUM = 550          # podłoga progu — dla towarów bez wyraźnej sezonowości
MARGINES = 1.5           # ile razy dłuższa niż zwykle musi być przerwa, żeby uznać za "martwy"

df_flagi = df_flagi.merge(prog_indywidualny, on='TowId', how='left')
df_flagi['PrognMartwy'] = np.maximum(
    N_MINIMUM,
    df_flagi['PrzerwaHistoryczna95pct'].fillna(0) * MARGINES
)

# 5) Flaga dynamiczna: czy W TYM DNIU przerwa od ostatniej sprzedaży przekroczyła próg.
#    Wiersze przed pierwszą sprzedażą towaru (DniOdSprzedazy = NaN) dostają False.
#    Uwaga: kalendarz jest przycięty do "ostatnia sprzedaż + 14 dni", więc dla towarów, które
#    przestały się sprzedawać, ta flaga zapala się tylko na ruchach (PZ/ST/...) po długiej przerwie.
df_flagi['CzyMartwyWTymDniu'] = df_flagi['DniOdSprzedazy'] >= df_flagi['PrognMartwy']

# 6) Flaga "na koniec danych": czy towar jest martwy na ostatni dzień danych
#    (dni od ostatniej sprzedaży do końca danych >= próg towaru). Ta sama wartość w każdym wierszu TowId.
data_max = df_flagi['Data'].max()
dni_do_konca = (data_max - ostatnia_sprzedaz).dt.days
prog_towid = df_flagi.drop_duplicates('TowId').set_index('TowId')['PrognMartwy']
martwy_na_koniec = dni_do_konca >= prog_towid.reindex(dni_do_konca.index)
df_flagi['CzyMartwyNaKoniecDanych'] = (
    df_flagi['TowId'].map(martwy_na_koniec).fillna(False).astype(bool)
)
df_flagi['CzyMartwyWTymDniu'] = df_flagi['CzyMartwyWTymDniu']
df_flagi['CzyMartwyNaKoniecDanych'] = df_flagi['CzyMartwyNaKoniecDanych'].astype('int8')

df_flagi['PrognMartwy'] = df_flagi['PrognMartwy'].astype('int32')
print("CzyMartwyWTymDniu (wiersze):")
print(df_flagi['CzyMartwyWTymDniu'].value_counts())
print("\nCzyMartwyNaKoniecDanych (unikalne TowId):")
print(df_flagi.drop_duplicates('TowId')['CzyMartwyNaKoniecDanych'].value_counts())
print(f"\nPróg indywidualny — opis:\n{df_flagi['PrognMartwy'].describe()}")


CzyMartwyWTymDniu (wiersze):
CzyMartwyWTymDniu
False    9429699
True       16704
Name: count, dtype: Int64

CzyMartwyNaKoniecDanych (unikalne TowId):
CzyMartwyNaKoniecDanych
0    9383
1    3098
Name: count, dtype: int64

Próg indywidualny — opis:
count    9.485872e+06
mean     5.531309e+02
std      3.598728e+01
min      5.500000e+02
25%      5.500000e+02
50%      5.500000e+02
75%      5.500000e+02
max      1.420000e+03
Name: PrognMartwy, dtype: float64


In [68]:

# ============================================================
# CzySezonowy — informacyjna flaga (niezależna od CzyMartwy)
# ============================================================

# Zobaczmy najpierw rozkład, żeby próg nie był z sufitu
print(df_flagi.drop_duplicates('TowId')['PrzerwaHistoryczna95pct'].describe())
print(df_flagi.drop_duplicates('TowId')['PrzerwaHistoryczna95pct'].quantile([0.5, 0.75, 0.9, 0.95, 0.99]))

count    11846.000000
mean        44.195003
std         80.849101
min          1.000000
25%          7.950000
50%         16.000000
75%         39.937500
max        947.000000
Name: PrzerwaHistoryczna95pct, dtype: float64
0.50     16.0000
0.75     39.9375
0.90    109.8000
0.95    199.0000
0.99    400.6025
Name: PrzerwaHistoryczna95pct, dtype: float64


In [69]:
# CzySezonowy = 1, gdy typowa przerwa w sprzedaży (95 percentyl) wynosi >= PROG_SEZONOWY dni
PROG_SEZONOWY = 100  # dni — zgodny z wcześniejszym prog_zagrozony, tuż przy 90. percentylu (109.8)

towid_sezonowe = (
    df_flagi.drop_duplicates('TowId')
    .loc[lambda d: d['PrzerwaHistoryczna95pct'] >= PROG_SEZONOWY, 'TowId']
)

df_flagi['CzySezonowy'] = df_flagi['TowId'].isin(towid_sezonowe)

print(df_flagi.drop_duplicates('TowId')['CzySezonowy'].value_counts())

CzySezonowy
False    11184
True      1297
Name: count, dtype: int64


In [70]:
# Flagi jako int8 (0/1): mniejszy plik i bezproblemowy zapis do parquet; podgląd jednego wiersza
df_flagi['CzySezonowy'] = df_flagi['CzySezonowy'].astype('int8')
df_flagi['CzyMartwyWTymDniu'] = df_flagi['CzyMartwyWTymDniu']
df_flagi['CzyMartwyNaKoniecDanych'] = df_flagi['CzyMartwyNaKoniecDanych']


In [71]:
df_flagi.rename(columns={'JestWazony': 'CzyWazony'}, inplace=True)

In [72]:
# ============================================================
# Kategoria ABC(D) — Pareto na wartości sprzedaży, per TowId
# ============================================================

# 1) Suma wartości SPRZEDAŻY per TowId (tylko TypDok == 21; ruchy PZ/BO/REM i puste dni nie wchodzą do sumy)
jest_sprzedaz = (df_flagi['DokId'] != -1) & (df_flagi['TypDok'] == 21)

wartosc_towid = (
    df_flagi.loc[jest_sprzedaz].groupby('TowId')['Wartosc'].sum()
    .reindex(df_flagi['TowId'].unique(), fill_value=0)
    .rename_axis('TowId')
    .reset_index()
    .sort_values('Wartosc', ascending=False)
    .reset_index(drop=True)
)

wartosc_towid['WartoscSkumulowanaPct'] = (
    wartosc_towid['Wartosc'].cumsum() / wartosc_towid['Wartosc'].sum() * 100
)

# 2) Czysty podział Pareto — bez uwzględniania martwych
def przypisz_abc(pct):
    if pct <= 80:
        return 'A'
    elif pct <= 95:
        return 'B'
    else:
        return 'C'

wartosc_towid['KategoriaABC'] = wartosc_towid['WartoscSkumulowanaPct'].apply(przypisz_abc)

print("Czysty podział Pareto (bez D):")
print(wartosc_towid['KategoriaABC'].value_counts())

# 3) Praktyczny wariant ABCD — martwe na koniec danych nadpisują D (kandydaci do wycofania)
mapa_martwy = df_flagi.drop_duplicates('TowId').set_index('TowId')['CzyMartwyNaKoniecDanych']
wartosc_towid['CzyMartwy'] = wartosc_towid['TowId'].map(mapa_martwy)

wartosc_towid['KategoriaABCD'] = wartosc_towid['KategoriaABC']
wartosc_towid.loc[wartosc_towid['CzyMartwy'] == 1, 'KategoriaABCD'] = 'D'

print("\nPraktyczny podział ABCD (martwe → D):")
print(wartosc_towid['KategoriaABCD'].value_counts())

# 4) Które D-ki BYŁY ważne (A lub B), zanim umarły
wartosc_towid['AlarmMartwe'] = (
    (wartosc_towid['KategoriaABCD'] == 'D') &
    (wartosc_towid['KategoriaABC'].isin(['A', 'B']))
)

print(f"\nAlarm — dawne A/B, teraz martwe: {wartosc_towid['AlarmMartwe'].sum()}")


for kol in ['KategoriaABC', 'KategoriaABCD', 'AlarmMartwe']:
    mapa = wartosc_towid.set_index('TowId')[kol]
    df_flagi[kol] = df_flagi['TowId'].map(mapa)

Czysty podział Pareto (bez D):
KategoriaABC
C    6620
B    3084
A    2777
Name: count, dtype: int64

Praktyczny podział ABCD (martwe → D):
KategoriaABCD
C    3955
D    3098
A    2727
B    2701
Name: count, dtype: int64

Alarm — dawne A/B, teraz martwe: 433


In [73]:
df_flagi['CzyMartwyWTymDniu'].value_counts()

CzyMartwyWTymDniu
False    9429699
True       16704
Name: count, dtype: Int64

In [74]:
df_flagi['CzyMartwyWTymDniu']=df_flagi['CzyMartwyWTymDniu'].fillna(False).astype('int8')
df_flagi['AlarmMartwe']=df_flagi['AlarmMartwe'].fillna(False).astype('int8')
df_flagi.dtypes.to_frame('Typ').rename_axis('Kolumna').reset_index()

,Kolumna,Typ
0,TowId,int64
1,Data,datetime64[ns]
2,DokId,int32
3,Kolejnosc,int32
4,NrPozycji,int32
5,TypPoz,int32
6,IloscPlus,float64
7,IloscMinus,float64
8,CenaPoRab,float64
9,Wartosc,float64


## Promocja

In [75]:
# tylko sprzedaż, cena > 0
sprz = df_flagi[(df_flagi['DokId'] != -1) & (df_flagi['TypDok'] == 21) & (df_flagi['CenaPoRab'] > 0)]

# cena towaru w danym dniu (mediana z paragonów tego dnia)
ceny = (sprz.groupby(['TowId', 'Data'], as_index=False)['CenaPoRab'].median()
            .sort_values(['TowId', 'Data']))

OKNO = '121D'   # okno czasowe: 60 dni wstecz i 60 wprzód (zamiast 60 sprzedaży)
PROG = 0.05     # obniżka od 5 % = promocja

def cena_zwykla(g):
    x = g.set_index('Data')['CenaPoRab']
    return pd.Series(x.rolling(OKNO, min_periods=5, center=True).median().values, index=g.index)

ceny['CenaZwykla'] = ceny.groupby('TowId', group_keys=False).apply(cena_zwykla)
ceny['Obnizka'] = 1 - ceny['CenaPoRab'] / ceny['CenaZwykla']
ceny['CzyPromocja'] = (ceny['Obnizka'] >= PROG).astype('int8')

# z powrotem do kalendarza (puste dni i ruchy dostają 0)
df_flagi = df_flagi.merge(
    ceny[['TowId', 'Data', 'CenaZwykla', 'Obnizka', 'CzyPromocja']],
    on=['TowId', 'Data'], how='left')
df_flagi['CzyPromocja'] = df_flagi['CzyPromocja'].fillna(0).astype('int8')

In [76]:
df_flagi.sample(5)

,TowId,Data,DokId,Kolejnosc,NrPozycji,TypPoz,IloscPlus,IloscMinus,CenaPoRab,Wartosc,...,PrognMartwy,CzyMartwyWTymDniu,CzyMartwyNaKoniecDanych,CzySezonowy,KategoriaABC,KategoriaABCD,AlarmMartwe,CenaZwykla,Obnizka,CzyPromocja
4198243,23591,2025-08-18,-1,0,0,4,0.0,0.0,9.5143,0.0,...,550,0,0,0,A,A,0,NaN,NaN,0
372069,1162,2023-11-25,-1,0,0,4,0.0,0.0,18.2846,0.0,...,550,0,0,0,A,A,0,NaN,NaN,0
6881246,53865,2023-08-04,-1,0,0,4,0.0,0.0,1.8426,0.0,...,550,0,0,1,C,C,0,NaN,NaN,0
7914437,66147,2023-09-11,-1,0,0,4,0.0,0.0,13.9900,0.0,...,550,0,1,0,B,D,1,NaN,NaN,0
413544,1603,2023-05-19,-1,0,0,4,0.0,0.0,2.8374,0.0,...,550,0,1,0,B,D,1,NaN,NaN,0


In [77]:
# Zapis wyniku (kalendarz + wszystkie flagi) do dane/interim
nazwa_pliku = "fact_inka_promocja.parquet"
df_flagi.to_parquet(
    f"dane/interim/{nazwa_pliku}",
    compression='zstd',
    index=False
)
print(f"Zapisano: {df_flagi.shape}")

Zapisano: (9485872, 44)


In [78]:
# Suma kontrolna wyniku — przelicz po każdej zmianie kodu

moj_hash = tools.hash_danych_bezpieczny(f"dane/interim/{nazwa_pliku}")
print(f"Mój hash (posortowane):   {nazwa_pliku}   {moj_hash}")

Mój hash (posortowane):   fact_inka_promocja.parquet   e99737271639d6c7b4a17e76d697a33f1c38009a1c77006e0c2b795a4993f77a


**fact_inka_final.parquet   c75c1673ab7c465ba51a09665d2a60e4bd59623d33bba30cbea8c7c8f7fe36b2**